In [20]:
import pandas as pd
file_path = r"/Users/suppi/Desktop/santosh/santosh/medical_extraction/Export_Commodity_Top100.xlsx"
xl = pd.read_excel(file_path, header=9)

In [10]:
header = xl.columns.tolist()

In [38]:
x = "| ".join(header)
x

'S.No.| HSCode| Commodity| 2023-2024| %Share| 2024-2025(Apr-Aug) | %Share.1| %Growth| HS Code digit level option'

In [42]:
chunks = []
for index, row in xl.iterrows():
    y = row.tolist()
    row_str = " | ".join(map(str, y))
    chunk = x + "\n" + row_str
    chunks.append(chunk)

In [43]:
print(len(chunks))

102


In [52]:
from langchain_openai import AzureOpenAIEmbeddings, AzureChatOpenAI
from langchain_core.documents import Document
from langchain.vectorstores import FAISS
from dotenv import load_dotenv
load_dotenv()

True

In [55]:
embeddings_chunks = [Document(page_content= chunk) for chunk in chunks]

In [56]:
embeddings_chunks

[Document(metadata={}, page_content='S.No.| HSCode| Commodity| 2023-2024| %Share| 2024-2025(Apr-Aug) | %Share.1| %Growth| HS Code digit level option\n1 | 1.0 | LIVE ANIMALS. \xa0  | 10246.86 | 0.0028 | 7818.25 | 0.0052 | -23.701016701701793 | 4 6 8 '),
 Document(metadata={}, page_content='S.No.| HSCode| Commodity| 2023-2024| %Share| 2024-2025(Apr-Aug) | %Share.1| %Growth| HS Code digit level option\n2 | 2.0 | MEAT AND EDIBLE MEAT OFFAL. \xa0  | 3174669.89 | 0.8772 | 1248846.77 | 0.8374 | -60.66215344361363 | 4 6 8 '),
 Document(metadata={}, page_content='S.No.| HSCode| Commodity| 2023-2024| %Share| 2024-2025(Apr-Aug) | %Share.1| %Growth| HS Code digit level option\n3 | 3.0 | FISH AND CRUSTACEANS, MOLLUSCS AND OTHER AQUATIC INVERTABRATES. \xa0  | 5071041.86 | 1.4012 | 1953452.6 | 1.3098 | -61.47827894286008 | 4 6 8 '),
 Document(metadata={}, page_content="S.No.| HSCode| Commodity| 2023-2024| %Share| 2024-2025(Apr-Aug) | %Share.1| %Growth| HS Code digit level option\n4 | 4.0 | DAIRY PROD

In [53]:
llm = AzureChatOpenAI(
    azure_deployment="gpt-4o",
    api_version= "2025-01-01-preview"
)

embeddings = AzureOpenAIEmbeddings(
    azure_deployment = "text-embedding-3-large",
    api_version= '2024-12-01-preview'
)

In [64]:
def build_vectorstore(docs, path = r"./faiss_index"):
    vectorstore = FAISS.from_documents(docs, embeddings)
    vectorstore.save_local(path)
    return vectorstore

In [65]:
vector_store = build_vectorstore(embeddings_chunks)

In [66]:
retieved_chunks = vector_store.similarity_search("How many imports are there in related water bodies", k = 5)

In [69]:
context = "\n\n".join([chunk.page_content for chunk in retieved_chunks])

In [72]:
print(context)

S.No.| HSCode| Commodity| 2023-2024| %Share| 2024-2025(Apr-Aug) | %Share.1| %Growth| HS Code digit level option
3 | 3.0 | FISH AND CRUSTACEANS, MOLLUSCS AND OTHER AQUATIC INVERTABRATES.    | 5071041.86 | 1.4012 | 1953452.6 | 1.3098 | -61.47827894286008 | 4 6 8 

S.No.| HSCode| Commodity| 2023-2024| %Share| 2024-2025(Apr-Aug) | %Share.1| %Growth| HS Code digit level option
88 | 89.0 | SHIPS, BOATS AND FLOATING STRUCTURES.    | 3359479.71 | 0.9283 | 1671668.76 | 1.1209 | -50.240248362744246 | 4 6 8 

S.No.| HSCode| Commodity| 2023-2024| %Share| 2024-2025(Apr-Aug) | %Share.1| %Growth| HS Code digit level option
16 | 16.0 | PREPARATIONS OF MEAT, OF FISH OR OF CRUSTACEANS, MOLLUSCS OR OTHER AQUATIC INVERTEBRATES    | 600929.83 | 0.1661 | 243965.04 | 0.1636 | -59.40207528056977 | 4 6 8 

S.No.| HSCode| Commodity| 2023-2024| %Share| 2024-2025(Apr-Aug) | %Share.1| %Growth| HS Code digit level option
nan | nan | India's Total Export | 361895227.05 | nan | 149136120.3 | nan |    | nan

S.No.| HS

In [47]:
print(llm.invoke("Hi?").content)

Hello! How can I assist you today?


# RAG

In [ ]:
df_markdown = xl.to_markdown(index=False) if xl is not None and not xl.empty else "No structured table available."

In [82]:
SYSTEM_PROMPT = """
You are a senior trade and economics research analyst specializing in India's import patterns.
You must:
1. Base every statement on provided evidence (tables, retrieved documents).
2. Never fabricate figures; if data is missing, state it clearly.
3. When numerical, calculate totals/percentages from the given data.
4. Present answers in this format:
   ## Summary
   <2–4 sentences high-level finding>

   ## Key Figures
   <Markdown table>

   ## Insights & Hypotheses
   - Bullet points with trends, comparisons, anomalies

   ## Next Steps
   - Optional recommendations for further analysis
5. Be objective and concise.
6. Only use Markdown.
"""


In [83]:
def QandA(question):
    prompt = f"""{SYSTEM_PROMPT}
    # Structured Data Table
    {df_markdown}

    # Question
    {question}
    """
    resp = llm.invoke(prompt).content
    return resp

In [86]:
while True:
    question = input("Ask your question")
    if question :
        ans = QandA(question)
        print(ans)
    else :
        break

## Summary
The import data shows significant reductions in various import categories year-on-year, with some sectors experiencing less decline than others. This provides insight into shifting demand and potential areas of investment. Categories related to mineral fuels, machinery, and pharmaceuticals are prominent in terms of value, despite a general decline.

## Key Figures
| Category                                                              | 2023-2024 (Value) | 2023-2024 (% Share) | 2024-2025 (Apr-Aug) (Value) | 2024-2025 (% Share) | % Growth           |
|:----------------------------------------------------------------------|------------------:|--------------------:|---------------------------:|--------------------:|:------------------|
| Mineral Fuels, Oils (HS 27)                                           |          7.25e+07 |              20.04% |                  2.82e+07  |         18.92%      | -61.10%           |
| Machinery, Nuclear Reactors (HS 84)                      

In [ ]:
FAISS_PATH = r"/Users/suppi/Desktop/santosh/santosh/medical_extraction/faiss_index"
def qanda(question):
    vector_store = FAISS.load_local(FAISS_PATH, embeddings,allow_dangerous_deserialization = True)
    retrieved_chunks = vector_store.similarity_search(question, k = 10)
    context = "\n\n".join([chunk.page_content for chunk in retrieved_chunks])
    prompt = f"You are an expert researcher based on t e"